In [ ]:
from pyspark.sql import functions as F

catalog = "retail_operations_dev"

customers = spark.table(
    f"{catalog}.silver.customers_current"
)

order_items = spark.table(
    f"{catalog}.gold.order_items_enriched"
)

duplicate_customer_id = (
    customers
    .groupBy('customer_id')
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_customer_id.limit(1).count() == 0,(
    "Duplicate customer IDs would multiply order lines during the join."
)

In [ ]:
customer_enriched_items = (
    order_items.alias('orders')
    .join(
        customers.alias("customers"),
        F.col("orders.customer_id") == F.col("customers.customer_id"),
        how = "left"
    )
    .select(
        "orders.*",
        F.col("customers.customer_name").alias("current_customer_name"),
        F.col("customers.country_code").alias("current_customer_country"),
        F.col("customers.customer_segment").alias("current_customer_segment"),
        F.col("customers.is_active").alias("customer_currently_active"),
        F.col("customers.updated_at").alias("customer_profile_updated_at"),
        F.col("customers.customer_id").isNull().alias("customer_missing")       
    )
)

display(
    customer_enriched_items
    .filter(F.col("customer_id") == "C001")
)

In [ ]:
original_columns = order_items.columns

original_fields_after_join = customer_enriched_items.select(
    *original_columns
)

assert(
    order_items
    .exceptAll(original_fields_after_join)
    .limit(1)
    .count() == 0
), "Some original order lines were lost or changed."

assert (
    original_fields_after_join
    .exceptAll(order_items)
    .limit(1).count() == 0
), "The join introduced extra or changed order lines."

display(
    customer_enriched_items
    .groupBy("customer_missing")
    .count()
)

In [ ]:
from delta.tables import DeltaTable

target_table = (
    f"{catalog}.gold.order_items_customer_enriched"
)

# Confirm that each order line appears only once.
duplicate_lines = (
    customer_enriched_items
    .groupBy("order_id", "line_id")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_lines.limit(1).count() == 0, (
    "Duplicate order lines found. Gold was not updated."
)

customer_enriched_items.createOrReplaceTempView(
    'customer_enriched_items_snapshot'
)

spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {target_table}
    USING DELTA
    AS
    SELECT *
    FROM customer_enriched_items_snapshot
    WHERE 1 = 0     
""")

# Synchronize Gold with the complete calculated result.

(
    DeltaTable.forName(spark, target_table)
    .alias("target")
    .merge(
        customer_enriched_items.alias("source"),
        """
        target.order_id = source.order_id
        AND
        target.line_id = source.line_id
        """
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .whenNotMatchedBySourceDelete()
    .execute()
)

In [ ]:
saved_items = spark.table(target_table).select(
    *customer_enriched_items.columns
)

assert (
    customer_enriched_items
    .exceptAll(saved_items)
    .limit(1).count() == 0
), "Some calculated rows are missing or different in Gold."

assert (
    saved_items
    .exceptAll(customer_enriched_items)
    .limit(1).count() == 0
), "Gold contains extra or different rows."

display(
    saved_items.select(
        "order_id",
        "line_id",
        "customer_id",
        "current_customer_segment",
        "product_id",
        "current_product_name",
        "quantity",
        "unit_price",
        "line_amount"
    ).orderBy("order_id", "line_id")
)

In [ ]:
enriched_lines = spark.table(
    f"{catalog}.gold.order_items_customer_enriched"
)

# Exclude cancelled orders from these metrics.
included_lines = enriched_lines.filter(
    F.col("order_status") != "cancelled"
)

# Aggregate the order facts before attaching customer attributes.

customer_metrics = (
    included_lines
    .groupBy("customer_id", "currency")
    .agg(
        F.countDistinct("order_id").alias("order_count"),
        F.count("*").alias("order_line_count"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("line_amount").alias("ordered_value")
    )
    .withColumn(
        "average_order_value",
        F.round(
            F.col("ordered_value") / F.col("order_count"),
            2
        )
    )
)

display(
    customer_metrics.orderBy(
        F.col("ordered_value").desc()
    )
)

In [ ]:
customer_summary = (
    customer_metrics.alias("metrics")
    .join(
        customers.alias("customers"),
        F.col("metrics.customer_id")
        == F.col("customers.customer_id"),
        "left"
    )
    .select(
        "metrics.*",
        F.col("customers.customer_name").alias("current_customer_name"),
        F.col("customers.country_code").alias("current_customer_country"),
        F.col("customers.customer_segment").alias("current_customer_segment"),
        F.col("customers.is_active").alias("customer_currently_active"),
        F.col("customers.customer_id").isNull().alias("customer_missing")
    )
)

assert customer_summary.count() == customer_metrics.count(), (
    "Joining customer profiles changed the number of summary rows."
)

display(
    customer_summary.orderBy(
        F.col("ordered_value").desc()
    )
)

In [ ]:
expected_totals = (
    included_lines
    .groupBy("currency")
    .agg(
        F.count("*").alias("order_line_count"),
        F.sum("quantity").alias("units_ordered"),
        F.sum("line_amount").alias("ordered_value")
    )
)

actual_totals = (
    customer_summary
    .groupBy("currency")
    .agg(
        F.sum("order_line_count").alias("order_line_count"),
        F.sum("units_ordered").alias("units_ordered"),
        F.sum("ordered_value").alias("ordered_value")
    )
)

assert (
    expected_totals.exceptAll(actual_totals)
    .limit(1).count() == 0
), "Customer summary totals do not match the original order lines."

assert (
    actual_totals.exceptAll(expected_totals)
    .limit(1).count() == 0
), "Customer summary contains unexpected totals."

duplicate_summary_keys = (
    customer_summary
    .groupBy("customer_id", "currency")
    .count()
    .filter(F.col("count") > 1)
)

assert duplicate_summary_keys.limit(1).count() == 0, (
    "More than one summary row exists for a customer and currency."
)

display(actual_totals)

In [ ]:
summary_table = f"{catalog}.gold.customer_order_summary"

(
    customer_summary.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(summary_table)
)

In [ ]:
saved_summary = spark.table(summary_table).select(
    *customer_summary.columns
)

assert (
    customer_summary.exceptAll(saved_summary)
    .limit(1).count() == 0
), "Some calculated summary rows are missing or different after saving."

assert (
    saved_summary.exceptAll(customer_summary)
    .limit(1).count() == 0
), "The saved summary contains extra or different rows."

display(
    saved_summary.orderBy(
        "currency",
        F.col("ordered_value").desc()
    )
)